# Q1 Embedding a query

In [1]:
from embedder import Embedder

2026-06-27 10:31:47.320905312 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
embedder_instance = Embedder()
embedded_text = embedder_instance.encode(text = "How does approximate nearest neighbor search work?")

In [ ]:
answer_q1 = embedded_text[0]
answer_q1

np.float64(-0.02058203437252893)

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

# Q2 Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

In [6]:
sqlitesearch_page = [
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
][0]

In [7]:
sqlitesearch_page

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

@ can be used to compute similarity here, only because the vectors are already properly normalized

In [8]:
sqlitesearch_page_embedded = embedder_instance.encode(text = sqlitesearch_page["content"])

# Compute the similarity between the two embeddings
answer_q2 = sqlitesearch_page_embedded @ embedded_text
answer_q2

np.float64(0.36107026789538205)

# Q3 Chunking and search by hand

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
chunk_texts = [c["content"] for c in chunks]
X = embedder_instance.encode_batch(chunk_texts)

In [11]:
v = embedder_instance.encode(text="How does approximate nearest neighbor search work?")
scores = X.dot(v)

In [12]:
import numpy as np

best_idx = np.argmax(scores)

In [13]:
best_chunk = chunks[best_idx]
print(best_chunk)

{'start': 1000, 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one p

In [14]:
answer_q3 = best_chunk["filename"]
answer_q3

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4 Vector search with minsearch

In [15]:
from minsearch import VectorSearch

In [16]:

vs = VectorSearch()
vs.fit(X, payload=chunks)        # index your embedded chunks + their metadata

embedded_query_vector_q4 = embedder_instance.encode(text="What metric do we use to evaluate a search engine?")

answer_q4 = vs.search(embedded_query_vector_q4, num_results=5)  # v = embedded query vector, not raw text

In [17]:
answer_q4

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

# Q5 Text search vs vector search

In [18]:
from minsearch import Index

In [22]:
vs = VectorSearch()
vs.fit(X, payload=chunks)        # index your embedded chunks + their metadata

embedded_query_vector_q5 = embedder_instance.encode(text="How do I store vectors in PostgreSQL?")

vector_search_answer = vs.search(embedded_query_vector_q5, num_results=5)  # v = embedded query vector, not raw text

text_search_index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
text_search_index.fit(chunks)
text_search_answer = text_search_index.search("How do I store vectors in PostgreSQL?", num_results=5)

for val in vector_search_answer:
    print(val["filename"])

print("\n\n")

for val in text_search_answer:
    print(val["filename"])


02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md



02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


# Q6 Hybrid search

In [20]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
question_q6 = "How do I give the model access to tools?"

embedded_query_vector_q6 = embedder_instance.encode(text= question_q6)

vector_search_answer = vs.search(embedded_query_vector_q6, num_results=5)  # v = embedded query vector, not raw text

text_search_index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
text_search_index.fit(chunks)
text_search_answer = text_search_index.search(question_q6, num_results=5)

results = rrf([vector_search_answer, text_search_answer])
answer_q6 = results[0]["filename"]

answer_q6

'01-agentic-rag/lessons/13-function-calling.md'

: 